# Round 13: forests on the strongest representation - S6E10

Round 12's Exp 36 (counts + leak-free TE + 3 seeds x {63, 127} leaves, 6-model average) became
the new preferred candidate: mean OOF 0.959996 (prints 0.96000; not strictly at the 0.960
target), +0.00014 vs the Exp 31 bar with all five fold deltas positive. Exp 36 also showed
something useful: config diversity, null on the counts-only base, measured positive on the TE
base - a more diverse model mix can earn its place on this representation.

Andrea's standing preference is random forests, and the standing brief is feature engineering
for all models. The forests were last measured on much weaker features (raw in Exp 24, counts
in Exp 28 - both clearly below the booster). This round gives them the strongest current
representation, one change of model family:

- **Exp 37: RandomForest on counts + TE.** The measured RF reference config (Exp 4: 400 trees,
  min_samples_leaf=5, seed 42) on fold-local counts + split-first leak-free cross-fitted TE.
  Scheme differences vs the booster lineage, disclosed: single seed (no averaging - forests are
  already bagged), the 4 native categoricals as ordinal codes, fold-local median imputation for
  the 292 delay NaNs (forests cannot take NaN natively - same handling as Exp 24).
- **Exp 38: ExtraTrees on counts + TE.** Same representation, ExtraTrees 400/leaf5. Measured
  worse on raw (Exp 24) and counts-only (Exp 28) features; retested here because the base
  changed. Family not written off; bounded to this config.

Bar and reading rules: the bar is Exp 36's round-12 fold constants, paired per-fold deltas,
2x-spread band as a decision heuristic (never proof). The LightGBM incumbent is recomputed
in-run as the determinism reference. Refit fires only on a working-signal win over the Exp 36
bar. The 0.960 target is a mean-OOF target at full precision - 0.959996 did not reach it.

In [ ]:
import inspect
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E29_FOLDS_R9 = [0.95938, 0.95804, 0.96033, 0.95933, 0.95956]        # counts + stochastic, round 9
E31_FOLDS_R10 = [0.95998, 0.95873, 0.96056, 0.96000, 0.96000]       # counts + TE, round 10
E36_FOLDS_R12 = [0.96016, 0.95887, 0.96067, 0.96009, 0.96019]       # counts + TE + config mix, the bar
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100
RF_PARAMS = dict(n_estimators=400, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE)

In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file():
    if Path('playground-series-s6e10/data/train.csv').is_file():
        competition_dir = Path('playground-series-s6e10')
    elif Path('../data/train.csv').is_file():
        competition_dir = Path('..')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding - not through the
    # value table and not through the smoothing mean (recomputed per inner fold from itr rows).
    # Inner folds are label-blind (plain shuffled KFold). (Round-10 audit-fixed construction.)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        # LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
        # use whichever signature the installed version supports (no warning either way).
        if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
            model.fit(X_fit, y_fit, eval_X=X_es, eval_y=y_es, eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        else:
            model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def forest_prep(X_tr, y_tr, X_va):
    # counts + leak-free TE on the outer-train rows (no early-stop set in a forest - the TE
    # boundary that matters here is own-label exclusion, which te_crossfit enforces),
    # ordinal codes for the natives, fold-local median impute (forests cannot take NaN)
    X_tr, (X_va,) = add_counts(X_tr, [X_va])
    te_tr = te_crossfit(X_tr, y_tr, TE_COLS_ALL)
    te_va = te_apply(X_tr, y_tr, X_va, TE_COLS_ALL)
    X_tr = pd.concat([X_tr.reset_index(drop=True), te_tr.reset_index(drop=True)], axis=1)
    X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
    for col in CATEGORICAL:
        cats = pd.Categorical(X_tr[col])
        X_tr[col] = cats.codes
        X_va[col] = pd.Categorical(X_va[col], categories=cats.categories).codes
    med = X_tr.median(numeric_only=True)
    return X_tr.fillna(med), X_va.fillna(med)

def oof_forest(feat_frame, label, name, kind):
    oof = np.zeros(len(feat_frame))
    aucs = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        X_tr, X_va = forest_prep(X_tr, y_tr, X_va)
        cls = RandomForestClassifier if kind == 'rf' else ExtraTreesClassifier
        model = cls(**RF_PARAMS)
        model.fit(X_tr, y_tr)
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a)
        print(f'  {name} fold {fold}: AUC {a:.5f}')
    return oof, aucs

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')

## Reference: incumbent recompute

Same-run LightGBM incumbent with the determinism check, as every round.

In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc

## Exp 37: RandomForest on counts + TE

Exp 4's measured RF config (400 trees, min_samples_leaf=5, seed 42) on the round-12
representation: fold-local counts, split-first leak-free cross-fitted TE on all 21 columns,
ordinal-coded natives, fold-local median impute. Single seed - disclosed scheme difference vs
the averaged booster; a promising result here earns a seed-averaged follow-up.

In [ ]:
print('Exp 37: RandomForest 400/leaf5 on counts + leak-free TE, seed 42')
oof37, aucs37 = oof_forest(features, target, 'E37', 'rf')
print(f'Exp 37 fold AUCs: {[round(a, 5) for a in aucs37]}')
print(f'Exp 37 mean OOF AUC: {np.mean(aucs37):.5f} +/- {np.std(aucs37):.5f} | pooled: {roc_auc_score(target, oof37):.6f}')

## Exp 38: ExtraTrees on counts + TE

Same representation, ExtraTrees 400/leaf5, seed 42. Measured worse on raw (Exp 24) and
counts-only (Exp 28) features - retested because the base changed. Bounded to this config.

In [ ]:
print('Exp 38: ExtraTrees 400/leaf5 on counts + leak-free TE, seed 42')
oof38, aucs38 = oof_forest(features, target, 'E38', 'et')
print(f'Exp 38 fold AUCs: {[round(a, 5) for a in aucs38]}')
print(f'Exp 38 mean OOF AUC: {np.mean(aucs38):.5f} +/- {np.std(aucs38):.5f} | pooled: {roc_auc_score(target, oof38):.6f}')

## Results, verdicts, and refit guard

Each candidate pairs against the in-run incumbent, the Exp 29 folds, and the Exp 36 folds (the
bar). Refit fires only on a WORKING SIGNAL better vs the Exp 36 bar; on a tie or loss, Exp 36
and submission_round12.csv stand. A forest losing to the bar is bounded to this representation
and config - not a verdict on the family. Remember the scheme asymmetry when reading gaps:
single-seed forests vs a 6-model averaged booster.

In [ ]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label

results = {}
for tag, aucs, desc in [('37', aucs37, 'Exp 37: RandomForest on counts+TE'), ('38', aucs38, 'Exp 38: ExtraTrees on counts+TE')]:
    print(desc)
    m_, s_, lab = paired_verdict(aucs, INC_FOLDS)
    print(f'  vs incumbent (in-run):         mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_, s_, lab = paired_verdict(aucs, E29_FOLDS_R9)
    print(f'  vs Exp 29 folds (round 9):     mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_bar, s_bar, lab_bar = paired_verdict(aucs, E36_FOLDS_R12)
    print(f'  vs Exp 36 folds (round 12 bar): mean delta {m_bar:+.5f} (spread {s_bar:.5f}) - {lab_bar}')
    results[tag] = (m_bar, lab_bar, float(np.mean(aucs)))

winners = [k for k, (m, lab, _) in results.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda k: results[k][2]) if winners else None
print()
if win:
    print(f'REFIT GUARD: Exp {win} clears the Exp 36 bar under the paired band -> refit fires (exploratory, descriptive signal).')
else:
    print('REFIT GUARD: nothing cleared the Exp 36 bar under the paired band. Exp 36 remains the '
          'preferred candidate; submission_round12.csv is untouched.')

In [ ]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    X_full, X_test = forest_prep(features, target, test_features)
    cls = RandomForestClassifier if win == '37' else ExtraTreesClassifier
    model = cls(**RF_PARAMS)
    model.fit(X_full, target)
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': model.predict_proba(X_test)[:, 1]})
    sub.to_csv('submission_round13.csv', index=False)
    scheme = f'Exp {win}: ' + ('RandomForest' if win == '37' else 'ExtraTrees') + ' 400/leaf5 on counts + TE, seed 42'
    print(f'submission_round13.csv written: {len(sub):,} rows | {scheme}. '
          f'Local file only - NOT submitted; Andrea submits.')

In [ ]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round13_oof.npz',
         incumbent=oof_inc, exp37=oof37, exp38=oof38, target=target.to_numpy())
meta = {
    'round': 13,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configs': {
        'incumbent': {'note': 'Exp 8 LightGBM config, in-run reference'},
        'exp37': {'scheme': 'RandomForest 400/leaf5, counts + leak-free TE, ordinal natives, median impute, seed 42'},
        'exp38': {'scheme': 'ExtraTrees 400/leaf5, same representation'},
        'rf_params': {k: str(v) for k, v in RF_PARAMS.items()},
    },
    'fold_aucs': {'incumbent': INC_FOLDS, 'exp37': aucs37, 'exp38': aucs38},
}
(cache_dir / 'round13_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round13_oof.npz + round13_oof.json')

## What happens next

- If a forest beats the bar: the family earns its place on this representation in this test -
  the refit fires and a seed-averaged forest follow-up is the bounded next step.
- If a forest gets close (inside the band vs the bar): its OOF decorrelates from the booster's
  (different family, different inductive bias) - an OOF-correlation check tells us whether a
  blend is worth one bounded test, where earlier blends failed on highly correlated pairs.
- Both lost clearly: bounded to these two configs on this representation - not a verdict on
  the forest family (seed-averaged forests and other configs remain unmeasured; the booster
  lineage's averaging gains sit an order below the measured gap, so expectations are low).
  Open levers on existing data: a robustness checkpoint re-scoring the leader under a second
  fold partition before more same-partition tuning, then informed bounded batches. External
  data stays gated on Andrea's explicit OK.